# 02 · Volatility Surface Construction

Load the snapshot from notebook 01, calibrate SVI smiles for every expiry,
and render the full 2D/3D implied-volatility surface.

**Run notebook 01 first** to generate the parquet files in `data/raw/`.

**Outputs:**
- `results/surface_3d.html` — interactive Plotly 3D surface
- `results/svi_smiles.png`  — per-expiry SVI fit quality
- `results/calibration_summary.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from src.surface_fit import VolSurface, SVIParams
from src.greeks import GreeksCalculator

Path('../results').mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Libraries loaded.')

In [ ]:
files = sorted(Path('../data/raw').glob('btc_surface_*.parquet'))
if not files:
    raise FileNotFoundError(
        'No BTC surface parquet files found in data/raw/. '
        'Run notebook 01_data_collection.ipynb first.'
    )

snap = pd.read_parquet(files[-1])
print(f'Loaded: {files[-1].name}')
print(f'  {len(snap)} options  |  {snap["expiry"].nunique()} expiries  |  {snap["type"].value_counts().to_dict()}')
print(f'  Spot: ${snap["spot"].iloc[0]:,.2f}')
print(f'  Timestamp: {snap["timestamp"].iloc[0]}')

In [ ]:
print('Fitting SVI surface (may take 1-2 min for many expiries)...')

surface = VolSurface(min_strikes=5)
surface.fit(snap, iv_col='calc_iv')

summ = surface.calibration_summary()
summ.to_csv('../results/calibration_summary.csv', index=False)

print(f'\nFitted {len(surface.slices)} slices')
print(f'Avg RMSE : {summ["rmse_ivpct"].mean():.3f} vol pts')
print(f'Max RMSE : {summ["rmse_ivpct"].max():.3f} vol pts  ({summ.loc[summ["rmse_ivpct"].idxmax(), "expiry"]})')
print(f'All arb-free: {summ["arb_free"].all()}')
print()
print(summ[['expiry', 'T_yr', 'n_strikes', 'rmse_ivpct', 'atm_vol', 'atm_skew', 'arb_free']].to_string(index=False))

In [ ]:
n_slices = len(surface.slices)
n_cols   = min(n_slices, 3)
n_rows   = (n_slices + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes_flat = np.array(axes).flatten()

k_grid = np.linspace(-0.55, 0.55, 300)

for ax, sl in zip(axes_flat, surface.slices):
    grp   = snap[snap['expiry'] == sl.expiry_str].dropna(subset=['calc_iv'])
    k_mkt = np.log(grp['strike'].values / sl.F)
    iv_mkt = grp['calc_iv'].values

    # Market data points — calls and puts separately
    for ot, marker, color in [('call', 'o', '#2E86AB'), ('put', '^', '#E84855')]:
        mask = grp['type'] == ot
        if mask.any():
            k_sub  = k_mkt[mask.values]
            iv_sub = iv_mkt[mask.values]
            ax.scatter(k_sub * 100, iv_sub * 100,
                       s=22, marker=marker, color=color, alpha=0.8,
                       zorder=5, label=ot)

    # SVI fit
    iv_fit = sl.params.implied_vol(k_grid, sl.T)
    ax.plot(k_grid * 100, iv_fit * 100, 'k-', lw=2, label='SVI fit', zorder=6)

    ax.set_title(
        f'{sl.expiry_str}  (T={sl.T*365:.0f}d)\n'
        f'ATM={sl.atm_vol()*100:.1f}%  RMSE={sl.calibration_rmse*100:.2f}vp',
        fontsize=9
    )
    ax.set_xlabel('Log-moneyness ×100', fontsize=8)
    ax.set_ylabel('IV (%)', fontsize=8)
    ax.legend(fontsize=7)

for ax in axes_flat[n_slices:]:
    ax.set_visible(False)

plt.suptitle('SVI Smile Fits — BTC Options', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/svi_smiles.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved results/svi_smiles.png')

In [ ]:
k_g, T_g, iv_m = surface.vol_grid(k_lo=-0.55, k_hi=0.55, n_strikes=100)

fig3d = go.Figure(data=[
    go.Surface(
        x=k_g * 100,
        y=T_g * 365.25,
        z=iv_m * 100,
        colorscale='RdYlBu_r',
        colorbar=dict(title='IV (%)', thickness=15, titlefont=dict(color='white')),
        contours=dict(
            z=dict(show=True, start=20, end=200, size=10,
                   color='white', width=1, usecolormap=False)
        ),
        hovertemplate=(
            'Log-moneyness: %{x:.1f}<br>'
            'Days to expiry: %{y:.0f}<br>'
            'IV: %{z:.1f}%<extra></extra>'
        ),
    )
])

fig3d.update_layout(
    title=dict(
        text=f'BTC Implied Volatility Surface (SVI) — Spot ${snap["spot"].iloc[0]:,.0f}',
        x=0.5, font=dict(color='white', size=14)
    ),
    scene=dict(
        xaxis=dict(title='Log-moneyness k×100', color='white', gridcolor='#444'),
        yaxis=dict(title='Days to Expiry', color='white', gridcolor='#444'),
        zaxis=dict(title='Implied Vol (%)', color='white', gridcolor='#444'),
        bgcolor='#0e1117',
        camera=dict(eye=dict(x=1.5, y=-1.8, z=0.8)),
    ),
    width=950,
    height=680,
    paper_bgcolor='#0e1117',
    plot_bgcolor='#0e1117',
    font=dict(color='white'),
    margin=dict(t=60, b=20, l=20, r=20),
)

fig3d.write_html('../results/surface_3d.html')
fig3d.show()
print('Saved results/surface_3d.html')

In [ ]:
print('=== NO-ARBITRAGE DIAGNOSTICS ===')

cal_viols = surface.check_calendar_arbitrage()
but_viols = surface.check_butterfly_arbitrage()

print(f'Calendar spread violations : {len(cal_viols)}')
if cal_viols:
    for v in cal_viols:
        print(f'  {v["expiry_lo"]} → {v["expiry_hi"]}:  '
              f'{v["n_violations"]} k-points, '
              f'max violation = {v["max_violation_var"]:.6f}')

print(f'Butterfly density violations: {len(but_viols)}')
if but_viols:
    for v in but_viols:
        print(f'  {v["expiry"]}:  {v["n_violations"]} k-points, '
              f'min density = {v["min_density"]:.6f}')

if not cal_viols and not but_viols:
    print('Surface is fully arbitrage-free.')

In [ ]:
print('=== GREEK SURFACE SAMPLE ===')

calc  = GreeksCalculator()
sl    = surface.slices[1] if len(surface.slices) > 1 else surface.slices[0]
F_ref = sl.F
T_ref = sl.T

strikes   = np.linspace(F_ref * 0.80, F_ref * 1.20, 9)
sample_rows = []

for K in strikes:
    k  = np.log(K / F_ref)
    iv = float(sl.params.implied_vol(np.array([k]), T_ref)[0])
    g  = calc.compute(F=F_ref, K=K, T=T_ref, sigma=iv, option_type='call')
    sample_rows.append({
        'Strike': f'${K:,.0f}',
        'Moneyness': f'{K/F_ref:.3f}',
        'IV (%)': f'{iv*100:.2f}',
        'Price ($)': f'{g.price:,.2f}',
        'Delta': f'{g.delta:.4f}',
        'Gamma': f'{g.gamma:.6f}',
        'Vega ($)': f'{g.vega:,.0f}',
        'Theta ($/day)': f'{g.theta:,.2f}',
    })

print(f'Expiry: {sl.expiry_str}  |  Forward: ${F_ref:,.2f}  |  T={T_ref*365:.0f}d')
print()
display(pd.DataFrame(sample_rows))